In [5]:
import pandas as pd
import h5py
import os
import numpy as np
from pandas.core import sample

In [7]:
df_path = 'dataframes/pediatric_imputed_survival.csv'
df = pd.read_csv(df_path)
df

,position,sample_name,slide_name,scan_name,tif_name,slide_id,exists,sample_name_alt,sample_type,sample_type_alt,...,Art_event,type_event,EFS_years,EFS_days,Death_disease,Date_of_death,OS_days,LastFollowup,date_of_LFU,Methylierungsgruppe
0,A2,1-T,T/1999/149,slide-2025-10-29T08-13-45-R1-S1.mrxs,slide-2025-10-29T08-13-45-R1-S1.ome.tif,slide-2025-10-29T08-13-45-R1-S1,slide-2025-10-29T08-13-45-R1-S1,1T,tumor,tumor,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2.0,2009-08-20 00:00:00,3
1,A3,2-b-T,T/2011/1450-b,slide-2025-10-29T08-25-47-R1-S2.mrxs,slide-2025-10-29T08-25-47-R1-S2.ome.tif,slide-2025-10-29T08-25-47-R1-S2,slide-2025-10-29T08-25-47-R1-S2,2bT,tumor,tumor,...,9,NaN,NaN,3162.0,1.0,NaN,3162.0,1.0,NaN,x
2,A6,2-d-T,T/2011/1450-d,slide-2025-10-29T08-35-06-R1-S3.mrxs,slide-2025-10-29T08-35-06-R1-S3.ome.tif,slide-2025-10-29T08-35-06-R1-S3,slide-2025-10-29T08-35-06-R1-S3,2dT,tumor,tumor,...,NaN,NaN,NaN,2318.0,1.0,NaN,2318.0,NaN,NaN,NaN
3,A8,3-T,T/2013/785,slide-2025-10-29T08-54-08-R1-S5.mrxs,slide-2025-10-29T08-54-08-R1-S5.ome.tif,slide-2025-10-29T08-54-08-R1-S5,slide-2025-10-29T08-54-08-R1-S5,3T,tumor,tumor,...,9,NaN,NaN,2555.0,1.0,NaN,2555.0,1.0,NaN,x
4,A9,4-T,T/2013/569,slide-2025-10-29T09-04-19-R1-S6.mrxs,slide-2025-10-29T09-04-19-R1-S6.ome.tif,slide-2025-10-29T09-04-19-R1-S6,slide-2025-10-29T09-04-19-R1-S6,4T,tumor,tumor,...,1.3. weitere Events . 25.02.14. 28.05.14,4.0,1.010959e+09,369.0,2.0,2015-07-01 00:00:00,901.0,2.0,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,G4,75-T,KT2008/1381,slide-2025-10-30T15-59-21-R2-S8.mrxs,slide-2025-10-30T15-59-21-R2-S8.ome.tif,slide-2025-10-30T15-59-21-R2-S8,slide-2025-10-30T15-59-21-R2-S8,75T,tumor,tumor,...,9,NaN,NaN,4651.0,1.0,NaN,4651.0,1.0,NaN,x
76,G5,76-T,KT/2008/1362-1,slide-2025-10-30T16-08-37-R2-S9.mrxs,slide-2025-10-30T16-08-37-R2-S9.ome.tif,slide-2025-10-30T16-08-37-R2-S9,slide-2025-10-30T16-08-37-R2-S9,76T,tumor,tumor,...,01/2009 1.Lokalrezidiv links mit bipulmonale M...,3.0,3.835616e-01,140.0,2.0,2010-08-28 00:00:00,729.0,2.0,2010-08-28 00:00:00,x
77,G6,77-T,KT/2008/1365,slide-2025-10-30T16-26-23-R2-S10.mrxs,slide-2025-10-30T16-26-23-R2-S10.ome.tif,slide-2025-10-30T16-26-23-R2-S10,slide-2025-10-30T16-26-23-R2-S10,77T,tumor,tumor,...,9,NaN,NaN,5121.0,1.0,NaN,5121.0,1.0,NaN,1
78,G8,78-T,KT/2008/956-1,slide-2025-10-30T16-36-42-R2-S11.mrxs,slide-2025-10-30T16-36-42-R2-S11.ome.tif,slide-2025-10-30T16-36-42-R2-S11,slide-2025-10-30T16-36-42-R2-S11,78T,tumor,tumor,...,9,NaN,NaN,NaN,1.0,,NaN,1.0,NaN,3


In [8]:
parent_dir = r"W:\pathologie\bioinfo-archive\UK_Augsburg_Claus\KristianUnger\TitanFeatures\20x_512px_0px_overlap\slide_features_titan"
h5_key = "features"

In [13]:
print("Extracting embeddings from .h5 files...")

data_list = []
seen_scans = set()  # Sets are O(1) for lookups vs O(n) for lists

for row in df.itertuples(): # itertuples is much faster than iterrows
    try:
        # Clean scan name
        base_name = row.scan_name.replace('.mrxs', '')

        # Avoid duplicates efficiently
        if base_name in seen_scans:
            continue

        pat_id = f"{row.ID_patient}{base_name.split('-')[-1]}"
        file_name = f"{base_name}.h5"
        h5_path = os.path.join(parent_dir, file_name)
        sample_name = row.sample_name

        if os.path.exists(h5_path):
            with h5py.File(h5_path, 'r') as f:
                vector = f[h5_key][:]
                if vector.ndim > 1:
                    vector = np.squeeze(vector)

                # Append a dictionary for each record
                data_list.append({
                    'scan_name': base_name,
                    'ID_patient': pat_id,
                    'embedding': vector,
                    'sample_name': sample_name,
                })
                seen_scans.add(base_name)

    except Exception as e:
        print(f"Error processing {row.scan_name}: {e}")

# Create the final DataFrame at once
new_df = pd.DataFrame(data_list)
# Convert the list of arrays into a separate DataFrame
embeddings_df = pd.DataFrame(new_df['embedding'].tolist())

# Rename columns to something like dim_0, dim_1...
embeddings_df.columns = [f'dim_{i}' for i in range(embeddings_df.shape[1])]

# Concatenate with your metadata and drop the original object column
final_csv_df = pd.concat([new_df.drop(columns=['embedding']), embeddings_df], axis=1)

# Save to CSV
final_csv_df.to_csv("embeddings.csv", index=False)
final_csv_df.head()

Extracting embeddings from .h5 files...


,scan_name,ID_patient,sample_name,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,dim_758,dim_759,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767
0,slide-2025-10-29T08-13-45-R1-S1,1S1,1-T,0.271729,0.442627,0.007004,-0.491455,-0.500000,0.662109,0.705566,...,-0.666016,0.947754,0.858887,0.197998,-0.048492,-0.061371,0.381104,0.110718,0.013802,0.503906
1,slide-2025-10-29T08-25-47-R1-S2,2S2,2-b-T,0.326416,0.114502,-0.427979,-0.391602,0.044373,0.682617,-0.316406,...,-0.469727,0.065308,0.644531,1.056641,0.559570,-0.018723,0.609863,0.325928,0.084167,0.067017
2,slide-2025-10-29T08-35-06-R1-S3,2S3,2-d-T,0.289062,0.007957,-0.188232,-0.314209,-0.187622,0.580566,-0.434326,...,-0.374756,0.054657,0.436279,1.191406,0.382324,0.020401,0.703125,0.234863,0.164062,-0.069824
3,slide-2025-10-29T08-54-08-R1-S5,3S5,3-T,0.007023,0.725098,-1.041992,0.367188,0.210815,0.152466,0.238037,...,-0.636230,0.783691,0.302490,-0.080383,0.433350,-0.615723,0.906738,-0.029953,0.029999,-0.135132
4,slide-2025-10-29T09-04-19-R1-S6,4S6,4-T,0.115479,0.237183,-1.044922,-0.284668,0.192749,0.766113,-0.244995,...,-0.631836,-0.050201,0.263184,0.804688,0.618164,-0.224609,0.515137,0.759277,0.261963,-0.434326
